In [43]:


# Base query URL from your request
endpoint = "https://services6.arcgis.com/clPWQMwZfdWn4MQZ/arcgis/rest/services/Person_Sev54/FeatureServer/0/query"

# SQL condition - adjust field name if Arizona is stored under 'State' or 'State_Name'
where_clause = "State = 'AZ' OR State = 'Arizona' OR State_Name = 'Arizona'"

all_records = []
offset = 0
record_limit = 1000

while True:
    params = {
        "where": "IncidentYear=2026",  # Use "1=1" if the layer only contains Arizona records
        "outFields":"*",
        "f": "json",  # Converts binary PBF to readable JSON
        "resultOffset": offset,
        "resultRecordCount": record_limit,
        "orderByFields": "OBJECTID ASC",
        "returnGeometry": "false"
    }

    response = requests.get(endpoint, params=params)
    data = response.json()
    
    features = data.get("features", [])
    if not features:
        break

    records = [f["attributes"] for f in features]
    all_records.extend(records)
        
    offset += record_limit
    print(f"Fetched {len(all_records)} records...")

# Save output to CSV
df = pd.DataFrame(all_records)
df.to_csv("arizona2026.csv", index=False)
print(f"Saved {len(df)} to .csv")

Saved 0 to .csv


In [1]:
import requests
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
fields = [
    "AccidentDateTime_Text",
    "IncidentID",
    "Latitude",
    "Longitude",
    "City",
    "County",
    "Onroad",
    "CrossingFeature",
    "IncidentFirstHarmfulEventDesc",
    "Sev5_count",
    "Sev4_count",
    "Sev3_count",
    "Sev2_count",
    "Sev1_count",
    "PersonInjuryStatusDesc",
    "PersonTypeDesc",
    "PersonSex",
    "PersonAge",
]
output = pd.DataFrame(columns=fields,data=[])
for year in range(2016,2026):
    print(year)
    data = pd.read_csv(f"arizona{year}.csv")
    data = data[fields]
    output = pd.concat([output, data], ignore_index=True)
    # break
output.to_csv("arizona_crash_data.csv")

2016
2017
2018
2019
2020
2021
2022
2023
2024
2025


In [7]:
persons = pd.read_csv(f"arizona_crash_data.csv").sort_values(by="IncidentID")

In [43]:
for year in range(2016,2026):
    persons = pd.read_csv("arizona_crash_data.csv").sort_values(by="IncidentID")
    crash_id =""
    for x in persons.index:
        crash_id = persons['IncidentID'][x]
        print(x)
        print(crash_id)
        if x > 10000:
            break

1834
3031363
5448
3031415
4884
3031498
3349
3031552
5451
3032095
1645
3032095
1661
3032219
3530
3032678
5422
3032867
2483
3032867
3727
3032903
853
3033270
5450
3033384
4886
3033395
2509
3033782
343
3033785
3666
3033838
3113
3033902
1344
3033921
3026
3033922
506
3034110
2300
3034120
1643
3034189
829
3034189
2885
3034244
41
3034244
825
3034251
54
3034251
3007
3034251
1308
3034251
2486
3034251
5437
3034279
2485
3034309
5353
3034310
2459
3034398
185
3034539
1193
3034697
432
3034703
1340
3034757
826
3035233
3029
3035282
100
3035390
1108
3035657
3754
3035790
5459
3035830
1209
3035936
3321
3036191
99
3036264
136
3036273
4915
3036301
1681
3036387
4
3036510
5286
3036516
4887
3036695
944
3036703
311
3036712
137
3036772
139
3037777
342
3037777
788
3037778
138
3038060
3494
3038063
1644
3038155
3531
3038217
3529
3038228
3
3038247
2105
3038260
2128
3038264
217
3038299
5460
3038299
344
3038355
1192
3038469
21
3038536
3492
3038573
1647
3038669
312
3038907
1309
3039029
214
3039105
4936
3039138
2484
303

In [46]:
persons.sort_values(by="AccidentDateTime_Text")['AccidentDateTime_Text'][5045]

'2016-01-01 01:31:00.000'

In [25]:
persons['PersonInjuryStatusDesc']

1834     SUSPECTED_SERIOUS_INJURY
5448     SUSPECTED_SERIOUS_INJURY
4884     SUSPECTED_SERIOUS_INJURY
3349     SUSPECTED_SERIOUS_INJURY
5451     SUSPECTED_SERIOUS_INJURY
                   ...           
49098                       FATAL
47394                       FATAL
46051    SUSPECTED_SERIOUS_INJURY
48616    SUSPECTED_SERIOUS_INJURY
47676    SUSPECTED_SERIOUS_INJURY
Name: PersonInjuryStatusDesc, Length: 49490, dtype: str

In [34]:
persons['Sev5_count'].unique()

array([nan,  1.,  2.,  5.,  4.,  3.,  7.,  6.])

In [36]:
persons[persons["IncidentID"]==3038228]

,Unnamed: 0,AccidentDateTime_Text,IncidentID,Latitude,Longitude,City,County,Onroad,CrossingFeature,IncidentFirstHarmfulEventDesc,Sev5_count,Sev4_count,Sev3_count,Sev2_count,Sev1_count,PersonInjuryStatusDesc,PersonTypeDesc,PersonSex,PersonAge
3529,3529,2016-01-01 05:25:00.000,3038228,33.457587,-112.327289,Avondale,Maricopa,I 010,M130,OVERTURN_ROLLOVER,NaN,1.0,NaN,NaN,NaN,SUSPECTED_SERIOUS_INJURY,DRIVER,M,21.0


In [47]:
from datetime import datetime, timezone
format_string = "%Y-%m-%d %H:%M:%S.%f"
dt = datetime.strptime("2016-01-01 01:31:00.000", format_string)
dt = dt.replace(tzinfo=timezone.utc)
dt

datetime.datetime(2016, 1, 1, 1, 31, tzinfo=datetime.timezone.utc)